## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✓ Libraries imported successfully")

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP
start_date = '2024-01-01'
end_date = '2024-03-31'

# Product filtering
# Common filter columns: jp_sub_category_alter_lang_name, jp_category_alter_lang_name, jp_brand_name
prod_filter_column = 'jp_sub_category_alter_lang_name'
prod_filter_value = '洗濯洗剤'  # Laundry detergent

# Granularity settings (hierarchical levels for aggregation)
# Options: jp_brand_name, jp_segment_name, jp_prod_name, jp_sub_category_name, jp_sub_brand_alter_lang_name
granularity_1 = 'jp_brand_alter_lang_name'      # Top level
granularity_2 = 'jp_segment_name'               # Middle level  
granularity_3 = 'jp_prod_name'                  # Bottom level

print(f"✓ Parameters configured")
print(f"  Customer: {customer_filter}")
print(f"  Period: {start_date} to {end_date}")
print(f"  Product filter: {prod_filter_column} = '{prod_filter_value}'")
print(f"  Granularity: {granularity_1} → {granularity_2} → {granularity_3}")

## 3. Build and Execute Query

In [ ]:
# Build C-TSR query with gold_customer_loyalty table
query = f"""
WITH tran_table AS (
    SELECT *
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw
),
prod_table AS (
    SELECT *
    FROM id_pos_ai_1.prod_dim_ext_vw
    WHERE {prod_filter_column} = '{prod_filter_value}'
),
site_ext_table AS (
    SELECT *
    FROM id_pos_ai_1.site_dim_ext_vw
),
shopper_table AS (
    SELECT *
    FROM id_pos_ai_1.shopper_dim_generic_vw
),
base_table AS (
    SELECT DISTINCT 
        '{customer_filter}' AS customer,
        CAST(t.sales_period_group_end_date_part AS DATE) AS date,
        YEAR(CAST(t.sales_period_group_end_date_part AS DATE)) AS TY_year,
        MONTH(CAST(t.sales_period_group_end_date_part AS DATE)) AS TY_month,
        YEAR(CAST(t.sales_period_group_end_date_part AS DATE)) - 1 AS YA_year,
        MONTH(CAST(t.sales_period_group_end_date_part AS DATE)) AS YA_month,
        p.{granularity_1} AS level1,
        p.{granularity_2} AS level2,
        p.{granularity_3} AS level3,
        t.pos_sales_amt AS value,
        t.pos_unit_sales_qty AS unit,
        t.shopper_key AS shopper_key,
        t.site_key AS site_key,
        s_ext.jp_store_format_desc AS store_format
    FROM tran_table t
    LEFT JOIN prod_table p ON t.prod_key = p.prod_key
    LEFT JOIN site_ext_table s_ext ON t.site_key = s_ext.site_key
    WHERE t.sales_period_group_end_date_part >= '{start_date}'
        AND t.sales_period_group_end_date_part <= '{end_date}'
        AND t.data_provider_code_part = '{customer_filter}'
),
ty AS (
    SELECT 
        customer,
        TY_year,
        TY_month,
        level1,
        level2,
        level3,
        SUM(value) AS ty_value,
        SUM(unit) AS ty_unit,
        COUNT(DISTINCT shopper_key) AS ty_shopper
    FROM base_table
    GROUP BY customer, TY_year, TY_month, level1, level2, level3
),
ya AS (
    SELECT 
        customer,
        YA_year AS TY_year,
        YA_month AS TY_month,
        level1,
        level2,
        level3,
        SUM(value) AS ya_value,
        SUM(unit) AS ya_unit,
        COUNT(DISTINCT shopper_key) AS ya_shopper
    FROM base_table
    WHERE date >= DATE_SUB(TO_DATE('{start_date}'), 365)
        AND date <= DATE_SUB(TO_DATE('{end_date}'), 365)
    GROUP BY customer, YA_year, YA_month, level1, level2, level3
),
ty_base AS (
    SELECT DISTINCT
        TY_year,
        TY_month,
        level1,
        level2,
        level3,
        shopper_key,
        site_key,
        store_format
    FROM base_table
),
ya_base AS (
    SELECT DISTINCT
        YA_year AS TY_year,
        YA_month AS TY_month,
        level1,
        level2,
        level3,
        shopper_key,
        site_key,
        store_format
    FROM base_table
    WHERE date >= DATE_SUB(TO_DATE('{start_date}'), 365)
        AND date <= DATE_SUB(TO_DATE('{end_date}'), 365)
)
SELECT 
    ty.customer,
    ty.level1,
    ty.level2,
    ty.level3,
    ty.ty_value,
    ya.ya_value,
    CASE WHEN ya.ya_value > 0 THEN (ty.ty_value / ya.ya_value) * 100 ELSE NULL END AS value_IYA,
    ty.ty_unit,
    ya.ya_unit,
    CASE WHEN ya.ya_unit > 0 THEN (ty.ty_unit / ya.ya_unit) * 100 ELSE NULL END AS unit_IYA,
    COUNT(DISTINCT ty_base.site_key) AS ty_store_count,
    COUNT(DISTINCT ya_base.site_key) AS ya_store_count,
    CASE 
        WHEN COUNT(DISTINCT ya_base.site_key) > 0 
        THEN (COUNT(DISTINCT ty_base.site_key) / COUNT(DISTINCT ya_base.site_key)) * 100 
        ELSE NULL 
    END AS closure_rate_IYA,
    ty.ty_shopper,
    ya.ya_shopper,
    CASE 
        WHEN ya.ya_shopper > 0 
        THEN (ty.ty_shopper / ya.ya_shopper) * 100 
        ELSE NULL 
    END AS shopper_IYA,
    CASE WHEN ty.ty_shopper > 0 THEN ty.ty_value / ty.ty_shopper ELSE 0 END AS ty_value_per_shopper,
    CASE WHEN ya.ya_shopper > 0 THEN ya.ya_value / ya.ya_shopper ELSE 0 END AS ya_value_per_shopper,
    CASE 
        WHEN (ya.ya_value / NULLIF(ya.ya_shopper, 0)) > 0 
        THEN ((ty.ty_value / NULLIF(ty.ty_shopper, 0)) / (ya.ya_value / NULLIF(ya.ya_shopper, 0))) * 100 
        ELSE NULL 
    END AS value_per_shopper_IYA,
    CASE WHEN ty.ty_unit > 0 THEN ty.ty_value / ty.ty_unit ELSE 0 END AS ty_avg_unit_price,
    CASE WHEN ya.ya_unit > 0 THEN ya.ya_value / ya.ya_unit ELSE 0 END AS ya_avg_unit_price,
    CASE 
        WHEN (ya.ya_value / NULLIF(ya.ya_unit, 0)) > 0 
        THEN ((ty.ty_value / NULLIF(ty.ty_unit, 0)) / (ya.ya_value / NULLIF(ya.ya_unit, 0))) * 100 
        ELSE NULL 
    END AS avg_unit_price_IYA
FROM ty
LEFT JOIN ya 
    ON ty.customer = ya.customer
    AND ty.TY_year = ya.TY_year
    AND ty.TY_month = ya.TY_month
    AND ty.level1 = ya.level1
    AND ty.level2 = ya.level2
    AND ty.level3 = ya.level3
LEFT JOIN ty_base 
    ON ty.TY_year = ty_base.TY_year
    AND ty.TY_month = ty_base.TY_month
    AND ty.level1 = ty_base.level1
    AND ty.level2 = ty_base.level2
    AND ty.level3 = ty_base.level3
LEFT JOIN ya_base 
    ON ty.TY_year = ya_base.TY_year
    AND ty.TY_month = ya_base.TY_month
    AND ty.level1 = ya_base.level1
    AND ty.level2 = ya_base.level2
    AND ty.level3 = ya_base.level3
GROUP BY 
    ty.customer, ty.level1, ty.level2, ty.level3,
    ty.ty_value, ya.ya_value,
    ty.ty_unit, ya.ya_unit,
    ty.ty_shopper, ya.ya_shopper
ORDER BY ty.ty_value DESC
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
numeric_columns = [
    'ty_value', 'ya_value', 'value_IYA',
    'ty_unit', 'ya_unit', 'unit_IYA',
    'ty_store_count', 'ya_store_count', 'closure_rate_IYA',
    'ty_shopper', 'ya_shopper', 'shopper_IYA',
    'ty_value_per_shopper', 'ya_value_per_shopper', 'value_per_shopper_IYA',
    'ty_avg_unit_price', 'ya_avg_unit_price', 'avg_unit_price_IYA'
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} rows")
print(f"  Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Calculate overall totals
overall_totals = {
    'Total TY Sales Value (¥)': df['ty_value'].sum(),
    'Total YA Sales Value (¥)': df['ya_value'].sum(),
    'Sales Value IYA (%)': (df['ty_value'].sum() / df['ya_value'].sum() * 100) if df['ya_value'].sum() > 0 else 0,
    'Total TY Units': df['ty_unit'].sum(),
    'Total YA Units': df['ya_unit'].sum(),
    'Units IYA (%)': (df['ty_unit'].sum() / df['ya_unit'].sum() * 100) if df['ya_unit'].sum() > 0 else 0,
    'Unique Products': len(df),
    'Total TY Shoppers': df['ty_shopper'].sum(),
    'Total YA Shoppers': df['ya_shopper'].sum(),
    'Shopper IYA (%)': (df['ty_shopper'].sum() / df['ya_shopper'].sum() * 100) if df['ya_shopper'].sum() > 0 else 0
}

print("=" * 60)
print("C-TSR ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nOverall Performance:")
for key, value in overall_totals.items():
    if 'Value' in key or 'Shopper' in key or 'Units' in key:
        print(f"  {key:.<45} {value:>15,.0f}")
    else:
        print(f"  {key:.<45} {value:>15,.2f}")

## 5. Visualizations

In [ ]:
# Top 10 products by TY sales value
top_10 = df.nlargest(10, 'ty_value')

fig = px.bar(
    top_10,
    x='ty_value',
    y='level1',
    orientation='h',
    title=f"Top 10 {granularity_1} by Sales Value",
    labels={'ty_value': 'Sales Value (¥)', 'level1': granularity_1},
    text='ty_value',
    color='value_IYA',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=100
)

fig.update_traces(texttemplate='¥%{text:,.0f}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=500,
    coloraxis_colorbar_title="Value IYA %"
)

fig.show()

In [ ]:
# YoY comparison: Value, Units, Shoppers
brand_summary = df.groupby('level1').agg({
    'ty_value': 'sum',
    'ya_value': 'sum',
    'ty_unit': 'sum',
    'ya_unit': 'sum',
    'ty_shopper': 'sum',
    'ya_shopper': 'sum'
}).reset_index().nlargest(8, 'ty_value')

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Sales Value', 'Units Sold', 'Unique Shoppers'),
    specs=[[{'type':'bar'}, {'type':'bar'}, {'type':'bar'}]]
)

# Sales Value
fig.add_trace(
    go.Bar(name='TY', x=brand_summary['level1'], y=brand_summary['ty_value'], marker_color='#4A90E2'),
    row=1, col=1
)
fig.add_trace(
    go.Bar(name='YA', x=brand_summary['level1'], y=brand_summary['ya_value'], marker_color='#E24A90'),
    row=1, col=1
)

# Units
fig.add_trace(
    go.Bar(name='TY', x=brand_summary['level1'], y=brand_summary['ty_unit'], marker_color='#4A90E2', showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Bar(name='YA', x=brand_summary['level1'], y=brand_summary['ya_unit'], marker_color='#E24A90', showlegend=False),
    row=1, col=2
)

# Shoppers
fig.add_trace(
    go.Bar(name='TY', x=brand_summary['level1'], y=brand_summary['ty_shopper'], marker_color='#4A90E2', showlegend=False),
    row=1, col=3
)
fig.add_trace(
    go.Bar(name='YA', x=brand_summary['level1'], y=brand_summary['ya_shopper'], marker_color='#E24A90', showlegend=False),
    row=1, col=3
)

fig.update_layout(
    title_text="Year-over-Year Performance Comparison (Top 8 Brands)",
    height=500,
    barmode='group'
)

fig.show()

In [ ]:
# IYA scatter plot: Value IYA vs Shopper IYA
scatter_data = df[df['value_IYA'].notna() & df['shopper_IYA'].notna()].copy()
scatter_data['bubble_size'] = scatter_data['ty_value'] / scatter_data['ty_value'].max() * 100

fig = px.scatter(
    scatter_data,
    x='shopper_IYA',
    y='value_IYA',
    size='bubble_size',
    color='value_per_shopper_IYA',
    hover_data=['level1', 'level2', 'ty_value', 'ty_shopper'],
    text='level1',
    title='Growth Matrix: Value IYA vs Shopper IYA',
    labels={
        'shopper_IYA': 'Shopper Count IYA (%)',
        'value_IYA': 'Sales Value IYA (%)',
        'value_per_shopper_IYA': 'Spend/Shopper IYA (%)'
    },
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=100
)

# Add quadrant lines at 100%
fig.add_hline(y=100, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=100, line_dash="dash", line_color="gray", opacity=0.5)

fig.update_traces(textposition='top center', textfont_size=8)
fig.update_layout(height=600)

fig.show()

## 6. Top Performers Analysis

In [ ]:
# Top 10 by TY value
print("Top 10 by This Year Sales Value:")
top_10_by_value = df.nlargest(10, 'ty_value')[[
    'level1', 'level2', 'level3', 'ty_value', 'value_IYA', 'ty_unit', 'unit_IYA', 'ty_shopper', 'shopper_IYA'
]]
top_10_by_value

In [ ]:
# Top growth products
print("\nTop 10 Growth Products (by Value IYA %):")
top_growth = df[df['value_IYA'].notna()].nlargest(10, 'value_IYA')[[
    'level1', 'level2', 'level3', 'ty_value', 'ya_value', 'value_IYA'
]]
top_growth

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"ctsr_analysis_{prod_filter_value}_{start_date}_to_{end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame(list(overall_totals.items()), columns=['Metric', 'Value'])
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Full data
    df.to_excel(writer, sheet_name='Full_Analysis', index=False)
    
    # Top performers
    top_10_by_value.to_excel(writer, sheet_name='Top_10_By_Value', index=False)
    top_growth.to_excel(writer, sheet_name='Top_Growth', index=False)

print(f"✓ Data exported to: {export_file}")